In [1]:
import duckdb, polars as pl, os, glob

DATA_PATH = "../data/raw/data.csv"
RAW_PQ = "../data/processed/data.parquet"
os.makedirs("../data/processed", exist_ok=True)

if not os.path.exists(RAW_PQ):
    duckdb.sql(f"""
    COPY (
      SELECT * FROM read_csv('{DATA_PATH}',
             header=true, sample_size=-1, ignore_errors=true, all_varchar=true)
    ) TO '{RAW_PQ}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
print("Parquet siap:", RAW_PQ)

Parquet siap: ../data/processed/data.parquet


In [2]:
raw_mb = sum(os.path.getsize(f) for f in glob.glob("../data/raw/*.csv")) / 1e6
pq_mb = os.path.getsize(RAW_PQ) / 1e6
n = pl.scan_parquet(RAW_PQ).select(pl.len()).collect().item()
print(f"CSV: {raw_mb:,.0f} MB | Parquet: {pq_mb:,.0f} MB | Baris: {n:,}")
print("Memenuhi syarat (>=500 MB atau >1 juta baris):", raw_mb >= 500 or n > 1_000_000)

CSV: 738 MB | Parquet: 273 MB | Baris: 32,735
Memenuhi syarat (>=500 MB atau >1 juta baris): True


In [3]:
lf = pl.scan_parquet(RAW_PQ)
schema = lf.collect_schema()
print(f"Jumlah kolom: {len(schema)}")
print(schema)
lf.head(5).collect()

Jumlah kolom: 11
Schema({'id': String, 'source': String, 'title': String, 'image': String, 'url': String, 'content': String, 'date': String, 'embedding': String, 'created_at': String, 'updated_at': String, 'summary': String})


id,source,title,image,url,content,date,embedding,created_at,updated_at,summary
str,str,str,str,str,str,str,str,str,str,str
"""83""","""tempo""","""Depo Plumpang Terbakar, Anggot…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Anggota Ko…","""2023-03-04 06:18:13+00""","""[-0.01590039,-0.034130897,0.00…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Anggota Komisi VII DPR RI Rofi…"
"""84""","""tempo""","""Jokowi Perintahkan Wapres Ma'r…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Presiden J…","""2023-03-04 06:04:38+00""","""[-0.017608976,-0.021786924,0.0…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Presiden Joko Widodo telah mem…"
"""85""","""tempo""","""HNW Mendukung Jamaah Umroh Fir…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Wakil Ketua MP…","""2023-03-04 06:18:04+00""","""[0.00841488,-0.023665192,0.006…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Wakil Ketua MPR RI Dr. H. M. H…"
"""86""","""tempo""","""Tim Dokkes Polri Telah Terima …","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Tim Kedokt…","""2023-03-04 06:44:10+00""","""[-0.012671886,-0.0039057182,0.…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Tim Kedokteran dan Kesehatan (…"
"""87""","""tempo""","""Bamsoet Ajak Komunitas Otomoti…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Ketua MPR RI s…","""2023-03-04 06:38:57+00""","""[-0.015486176,-0.0125719,-0.01…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Ketua MPR RI Bambang Soesatyo …"


In [4]:
nulls = lf.null_count().collect()
miss = (pl.DataFrame({"kolom": nulls.columns, "null": nulls.row(0)})
        .with_columns((pl.col("null") / n * 100).round(2).alias("persen_null"))
        .sort("persen_null", descending=True))
miss

kolom,null,persen_null
str,i64,f64
"""content""",441,1.35
"""image""",160,0.49
"""summary""",10,0.03
"""id""",0,0.0
"""source""",0,0.0
…,…,…
"""url""",0,0.0
"""date""",0,0.0
"""embedding""",0,0.0


In [5]:
card = lf.select([pl.col(c).n_unique().alias(c) for c in schema.names()]).collect()
card_df = pl.DataFrame({"kolom": card.columns, "n_unique": card.row(0)}).sort("n_unique")
print(card_df)

n_uniq_rows = lf.unique().select(pl.len()).collect().item()
print(f"\nBaris duplikat penuh: {n - n_uniq_rows:,} ({(n - n_uniq_rows)/n:.2%})")

shape: (11, 2)
┌────────────┬──────────┐
│ kolom      ┆ n_unique │
│ ---        ┆ ---      │
│ str        ┆ i64      │
╞════════════╪══════════╡
│ source     ┆ 7        │
│ created_at ┆ 3452     │
│ updated_at ┆ 3452     │
│ image      ┆ 26450    │
│ date       ┆ 31769    │
│ …          ┆ …        │
│ embedding  ┆ 32193    │
│ title      ┆ 32651    │
│ summary    ┆ 32726    │
│ id         ┆ 32735    │
│ url        ┆ 32735    │
└────────────┴──────────┘

Baris duplikat penuh: 0 (0.00%)


In [6]:
duckdb.sql(f"SUMMARIZE SELECT * FROM '{RAW_PQ}'").show(max_width=250)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────┬──────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────┬───────────────┬────────┬────────┬────────┬────────┬────────┬───────┬─────────────────┐
│ column_name │ column_type │                               min                                │                               max                                │ approx_unique │  avg   │  std   │  q25   │  q50   │  q75   │ count │ null_percentage │
│   varchar   │   varchar   │                             varchar                              │                             varchar                              │     int64     │ "null" │ "null" │ "null" │ "null" │ "null" │ int64 │  decimal(9,2)   │
├─────────────┼─────────────┼──────────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────────────────┼───────────────┼────────┼────────┼────────┼────────┼────────┼───────┼───────────────

In [7]:
str_cols = [c for c, t in schema.items() if t == pl.String]
issues = lf.select(
    [(pl.col(c).str.strip_chars() == "").sum().alias(c) for c in str_cols]
).collect()
pl.DataFrame({"kolom": issues.columns, "string_kosong": issues.row(0)}).filter(pl.col("string_kosong") > 0)

kolom,string_kosong
str,i64
